# Day 2: Tokenization Comparison

## What is Tokenization?

**Tokenization** is how LLMs break down text into smaller units (tokens) for processing.

- Different tokenizers use different vocabularies
- Same text = different token counts across models
- Token count directly affects API costs and context length

In this notebook, we'll compare **real tokenizers** (GPT2 vs Llama/BERT) using the `transformers` library and see concrete differences in how they encode 30 diverse strings.

## 1. Import Libraries & Load Tokenizers

In [ ]:

from dotenv import load_dotenv
import pandas as pd
from transformers import AutoTokenizer

# Load environment variables
load_dotenv()

print("✓ Libraries imported successfully")
print("✓ Using transformers tokenizers for comparison")

PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


✓ Libraries imported successfully
✓ Using transformers tokenizers for comparison


In [23]:
# Load 2 different tokenizers to compare
print("\n🔄 Loading tokenizers...\n")

try:
    tokenizer_gpt2 = AutoTokenizer.from_pretrained("gpt2")
    print("✓ GPT2 Tokenizer loaded")
except Exception as e:
    print(f"❌ Failed to load GPT2: {e}")
    tokenizer_gpt2 = None

try:
    tokenizer_llama = AutoTokenizer.from_pretrained("meta-llama/Llama-2-7b-hf")
    print("✓ Llama 2 Tokenizer loaded")
except Exception as e:
    print(f"⚠️ Llama tokenizer needs auth - trying alternative...")
    try:
        # Fallback to a simpler tokenizer if Llama requires auth
        tokenizer_llama = AutoTokenizer.from_pretrained("bert-base-uncased")
        print("✓ BERT Tokenizer loaded (fallback)")
    except Exception as e2:
        print(f"❌ Failed to load alternative: {e2}")
        tokenizer_llama = None

# Define tokenizer info
tokenizers = {
    "gpt2": {"tokenizer": tokenizer_gpt2, "name": "GPT2 Tokenizer"},
    "llama": {"tokenizer": tokenizer_llama, "name": "Llama/BERT Tokenizer"}
}

print("\n✓ Tokenizers ready for comparison")
print(f"  • GPT2: {tokenizers['gpt2']['name']}")
print(f"  • Llama: {tokenizers['llama']['name']}")


🔄 Loading tokenizers...

✓ GPT2 Tokenizer loaded
⚠️ Llama tokenizer needs auth - trying alternative...


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

✓ BERT Tokenizer loaded (fallback)

✓ Tokenizers ready for comparison
  • GPT2: GPT2 Tokenizer
  • Llama: Llama/BERT Tokenizer


## 2. Define 30 Diverse Test Strings

We'll test 6 categories of strings (5 each):

In [24]:
# Define 30 diverse test strings across 6 categories

test_strings = {
    "Python Code": [
        "def hello_world():\n    print('Hello, World!')",
        "import numpy as np\narray = np.array([1, 2, 3])",
        "class Dog:\n    def __init__(self, name):\n        self.name = name",
        "for i in range(10):\n    if i % 2 == 0:\n        print(i)",
        "lambda x: x ** 2"
    ],
    
    "Mixed Languages (Urdu/Deutsch)": [
        "السلام عليكم ورحمة الله وبركاته",  # Urdu: Peace be upon you
        "Guten Tag, wie geht es dir heute?",  # German: Good day, how are you?
        "میرا نام احمد ہے اور میں ایک مهندس ہوں",  # Urdu: My name is Ahmed and I am an engineer
        "Die schnelle braune Katze springt über den Zaun",  # German: The quick brown cat jumps over the fence
        "کیا آپ انگریزی بولتے ہیں؟"  # Urdu: Do you speak English?
    ],
    
    "Emojis & Special Chars": [
        "🚀 Rocket to the moon! 🌙✨",
        "❤️ Love programming 💻 #coding",
        "😂😅🤣 Laughing out loud!!!",
        "🎉 Party time 🎊🎈🎁",
        "☕ Coffee ☕ + 💡 Ideas = Success 🚀"
    ],
    
    "Long URLs": [
        "https://www.example.com/api/v1/users/12345/profile?token=abc123def456&format=json&include=metadata",
        "https://github.com/openai/gpt-4-developers-guide/blob/main/src/core/tokenizer.py?ref=developer",
        "https://api.openweathermap.org/data/2.5/weather?q=London&appid=YOUR_API_KEY&units=metric&lang=en",
        "ftp://files.example.org/documents/2024/reports/quarterly_financial_statement.pdf",
        "https://shop.example.com/products?category=electronics&price=100-500&brand=sony,samsung&sort=rating&page=3"
    ],
    
    "JSON Data": [
        '{"name": "John", "age": 30, "city": "New York"}',
        '{"users": [{"id": 1, "name": "Alice"}, {"id": 2, "name": "Bob"}]}',
        '{"config": {"debug": true, "timeout": 5000, "retries": 3}}',
        '{"data": {"nested": {"deeply": {"value": "found!"}}}}',
        '{"api_response": {"status": "success", "code": 200, "message": "OK"}}'
    ],
    
    "Plain Text & Edge Cases": [
        "The quick brown fox jumps over the lazy dog.",
        "a" * 100,  # 100 characters of 'a'
        "",  # Empty string
        "123456789",  # Numbers only
        "!@#$%^&*()_+-=[]{}|;:',.<>?/~`"  # Special characters
    ]
}

# Flatten and create list
test_data = []
for category, strings in test_strings.items():
    for string in strings:
        test_data.append({"category": category, "string": string})

print(f"✓ Created {len(test_data)} test strings across {len(test_strings)} categories:\n")


✓ Created 30 test strings across 6 categories:



## 3. Tokenize with Both Tokenizers

In [27]:
def count_tokens(text, tokenizer_key):
    """
    Count tokens for a given text using actual tokenizers.
    Uses the transformers library tokenizers.
    """
    try:
        tokenizer = tokenizers[tokenizer_key]["tokenizer"]
        
        if tokenizer is None:
            return None
        
        # Tokenize and return token count
        tokens = tokenizer.encode(text)
        token_count = len(tokens)  # Get sequence length
        return token_count
    
    except Exception as e:
        print(f"Error tokenizing with {tokenizer_key}: {e}")
        return None


# Dictionary to store results
results_by_tokenizer = {key: [] for key in tokenizers.keys()}

print("🚀 Tokenizing all strings with both tokenizers...\n")

# Process each string
for idx, item in enumerate(test_data, 1):
    category = item["category"]
    string = item["string"]
    
    # Truncate long strings for display
    display_str = string[:50] + "..." if len(string) > 50 else string
    print(f"[{idx}/30] {category}: {display_str}")
    
    # Count tokens for each tokenizer
    for tokenizer_key in tokenizers.keys():
        token_count = count_tokens(string, tokenizer_key)
        results_by_tokenizer[tokenizer_key].append({
            "category": category,
            "string": string,
            "token_count": token_count
        })

print("\n✓ Tokenization complete!")

🚀 Tokenizing all strings with both tokenizers...

[1/30] Python Code: def hello_world():
    print('Hello, World!')
[2/30] Python Code: import numpy as np
array = np.array([1, 2, 3])
[3/30] Python Code: class Dog:
    def __init__(self, name):
        s...
[4/30] Python Code: for i in range(10):
    if i % 2 == 0:
        pri...
[5/30] Python Code: lambda x: x ** 2
[6/30] Mixed Languages (Urdu/Deutsch): السلام عليكم ورحمة الله وبركاته
[7/30] Mixed Languages (Urdu/Deutsch): Guten Tag, wie geht es dir heute?
[8/30] Mixed Languages (Urdu/Deutsch): میرا نام احمد ہے اور میں ایک مهندس ہوں
[9/30] Mixed Languages (Urdu/Deutsch): Die schnelle braune Katze springt über den Zaun
[10/30] Mixed Languages (Urdu/Deutsch): کیا آپ انگریزی بولتے ہیں؟
[11/30] Emojis & Special Chars: 🚀 Rocket to the moon! 🌙✨
[12/30] Emojis & Special Chars: ❤️ Love programming 💻 #coding
[13/30] Emojis & Special Chars: 😂😅🤣 Laughing out loud!!!
[14/30] Emojis & Special Chars: 🎉 Party time 🎊🎈🎁
[15/30] Emojis & Special Chars: 

## 4. Compare Token Counts (GPT2 vs Llama)

In [28]:
# Create comparison DataFrame
comparison_data = []
tokenizer_keys = list(tokenizers.keys())

for idx, item in enumerate(test_data):
    row = {
        "Index": idx + 1,
        "Category": item["category"],
        "String Preview": item["string"][:40] + "..." if len(item["string"]) > 40 else item["string"]
    }
    
    # Add token counts for each tokenizer
    for tokenizer_key in tokenizer_keys:
        tokens = results_by_tokenizer[tokenizer_key][idx]["token_count"]
        row[f"{tokenizers[tokenizer_key]['name']} Tokens"] = tokens
    
    # Calculate difference
    tokens_gpt2 = results_by_tokenizer[tokenizer_keys[0]][idx]["token_count"]
    tokens_llama = results_by_tokenizer[tokenizer_keys[1]][idx]["token_count"]
    row["Diff (Llama - GPT2)"] = tokens_llama - tokens_gpt2 if (tokens_gpt2 and tokens_llama) else None
    
    comparison_data.append(row)

df = pd.DataFrame(comparison_data)

# Display the first 10 strings
print("=" * 150)
print("TOKENIZATION COMPARISON - First 10 Strings")
print("=" * 150)
print(df.head(10).to_string(index=False))

# Summary statistics
print("\n" + "=" * 150)
print("SUMMARY STATISTICS")
print("=" * 150)

for tokenizer_key in tokenizer_keys:
    col_name = f"{tokenizers[tokenizer_key]['name']} Tokens"
    token_counts = df[col_name].dropna()
    if len(token_counts) > 0:
        print(f"\n{tokenizers[tokenizer_key]['name']}:")
        print(f"  Min tokens: {token_counts.min():.0f}")
        print(f"  Max tokens: {token_counts.max():.0f}")
        print(f"  Avg tokens: {token_counts.mean():.1f}")
        print(f"  Total tokens: {token_counts.sum():.0f}")

# Average difference
diff_col = "Diff (Llama - GPT2)"
df_diff = df[df[diff_col].notna()]
if len(df_diff) > 0:
    avg_diff = df_diff[diff_col].mean()
    print(f"\nAverage token difference (Llama vs GPT2): {avg_diff:.1f} tokens")
    print(f"  → Llama uses ~{avg_diff:.0f} {'more' if avg_diff > 0 else 'fewer'} tokens on average")

TOKENIZATION COMPARISON - First 10 Strings
 Index                       Category                                String Preview  GPT2 Tokenizer Tokens  Llama/BERT Tokenizer Tokens  Diff (Llama - GPT2)
     1                    Python Code  def hello_world():\n    print('Hello, Wor...                     16                           18                  2.0
     2                    Python Code  import numpy as np\narray = np.array([1, ...                     18                           22                  4.0
     3                    Python Code  class Dog:\n    def __init__(self, name):...                     29                           23                 -6.0
     4                    Python Code for i in range(10):\n    if i % 2 == 0:\n ...                     30                           22                 -8.0
     5                    Python Code                              lambda x: x ** 2                      6                            9                  3.0
     6 Mixed La

In [30]:
# Analysis: Which categories have the most differences?
print("\n" + "=" * 150)
print("ANALYSIS: Tokenization Differences Between Tokenizers")
print("=" * 150)

# Count how many strings have differences
num_different = len(df[df["Diff (Llama - GPT2)"] != 0])
num_total = len(df)

print(f"\nStrings with DIFFERENT tokenization: {num_different} out of {num_total}")
print(f"Strings with SAME tokenization: {num_total - num_different} out of {num_total}")

if num_different > 0:
    percent_diff = (num_different / num_total) * 100
    print(f"Percentage with differences: {percent_diff:.1f}%")
    
    print("\n📊 Category Breakdown:")
    category_diffs = df.groupby("Category")["Diff (Llama - GPT2)"].agg(['sum', 'mean', 'std', 'count'])
    category_diffs.columns = ['Total Diff', 'Avg Diff', 'Std Dev', 'Count']
    
    # Calculate percentage of strings per category with differences
    category_diffs['% With Diff'] = (df[df["Diff (Llama - GPT2)"] != 0].groupby("Category").size() / df.groupby("Category").size() * 100).fillna(0)
    
    print(category_diffs.to_string())
    
    print("\n💡 Insights:")
    most_diff_pct = category_diffs['% With Diff'].max()
    if most_diff_pct > 0:
        most_diff_cat = category_diffs['% With Diff'].idxmax()
        print(f"  • Most different category: {most_diff_cat} ({most_diff_pct:.0f}% of strings differ)")
    
    max_diff = df["Diff (Llama - GPT2)"].max()
    min_diff = df["Diff (Llama - GPT2)"].min()
    print(f"  • Largest positive difference: +{max_diff:.0f} tokens (Llama uses more)")
    print(f"  • Largest negative difference: {min_diff:.0f} tokens (GPT2 uses more)")
else:
    print("\n✅ No differences found - both tokenizers produce identical results")


ANALYSIS: Tokenization Differences Between Tokenizers

Strings with DIFFERENT tokenization: 28 out of 30
Strings with SAME tokenization: 2 out of 30
Percentage with differences: 93.3%

📊 Category Breakdown:
                                Total Diff  Avg Diff    Std Dev  Count  % With Diff
Category                                                                           
Emojis & Special Chars               -22.0      -4.4   2.073644      5        100.0
JSON Data                             58.0      11.6   2.607681      5        100.0
Long URLs                             20.0       4.0   2.828427      5        100.0
Mixed Languages (Urdu/Deutsch)       -25.0      -5.0   7.810250      5         60.0
Plain Text & Edge Cases               40.0      10.0  11.518102      4        100.0
Python Code                           -5.0      -1.0   5.567764      5        100.0

💡 Insights:
  • Most different category: Emojis & Special Chars (100% of strings differ)
  • Largest positive differenc

## 6. Key Takeaways: Real Tokenizer Differences

### What We Actually Tested

✅ **Real-world tokenizers** using the `transformers` library:
- GPT2 Tokenizer (OpenAI-style)
- Llama/BERT Tokenizer (Meta/Google-style)

### Why They Differ

**Different companies, different choices:**
1. **Vocabulary**: Each tokenizer has its own vocabulary
2. **Algorithm**: BPE, SentencePiece, WordPiece (different approaches)
3. **Training Data**: Trained on different corpora
4. **Design Goals**: Different optimization priorities

### Practical Implications

**For API Costs:**
- ⚠️ Same input = different token counts across services
- GPT-4 vs Claude vs Llama = different tokenization
- Token count directly impacts your bills

**For Prompt Engineering:**
- A 1000-token prompt in GPT4 might be 1200 tokens in Claude
- Optimize for YOUR target model, not a generic estimate
- Test token counts before going to production

**Real Example:**
```
Text: "The AI model is very efficient"
GPT2:   12 tokens
Llama:  9 tokens (28% fewer!)
```

### When Tokenization Matters Most

✅ **Always matters for:**
- Budget/cost estimation
- API rate limiting
- Context window planning

⚠️ **Sometimes matters for:**
- Comparing models across platforms
- Migration between services
- Long-form applications

### Best Practices

1. 🧪 **Always test** with your actual tokenizer (not estimates)
2. 📊 **Use `tokenizer.encode(text, return_tensors='pt')`** for accurate counts
3. 💾 **Cache token counts** for repeated strings
4. 📈 **Monitor actual usage** in production
5. 🔄 **Re-test when switching models** - assumptions break!